In [ ]:
!pip install statsmodels

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from statsmodels.tsa.arima.model import ARIMA


# 1. Check that the shared metric function is available
if "calculate_metrics" not in globals():
    raise RuntimeError(
        "Run Code 7 first to define calculate_metrics()."
    )


# 2. Define paths and baseline settings
input_path = Path(
    "results/step_04/engineered_panel.csv"
)

target_path = Path(
    "results/step_05/test_target_identities.csv"
)

output_folder = Path(
    "results/step_13"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

FORECAST_HORIZON = 6
SEASONAL_PERIOD = 12
CROSTON_ALPHA = 0.1
ARIMA_ORDER = (1, 0, 0)


# 3. Load the panel and common test targets
panel = pd.read_csv(
    input_path,
    parse_dates=["Entry_Date"]
)

expected_targets = pd.read_csv(
    target_path
)

origin_values = (
    expected_targets["Origin_Date"].unique()
)

if len(origin_values) != 1:
    raise ValueError(
        "This code expects one common six-month test origin."
    )

origin_date = pd.Timestamp(
    origin_values[0]
)

references = sorted(
    expected_targets["Reference"].unique()
)

expected_horizons = sorted(
    expected_targets["Horizon"].unique()
)

if expected_horizons != list(
    range(1, FORECAST_HORIZON + 1)
):
    raise ValueError(
        "The test protocol must contain horizons 1 to 6."
    )


# 4. Restrict the forecasting inputs to available history
# Future test quantities are never used as baseline inputs.
historical_panel = panel.loc[
    panel["Entry_Date"] <= origin_date
].copy()

history = (
    historical_panel
    .pivot(
        index="Entry_Date",
        columns="Reference",
        values="Quantity"
    )
    .sort_index()
    .reindex(columns=references)
)

expected_history_dates = pd.date_range(
    start=history.index.min(),
    end=origin_date,
    freq="MS"
)

history = history.reindex(
    expected_history_dates
)

if history.isna().any().any():
    raise ValueError(
        "Missing historical observations were found. "
        "Missing months must not be silently replaced with zeros."
    )

if len(history) < SEASONAL_PERIOD:
    raise ValueError(
        "At least 12 historical months are required."
    )


# 5. Define the naive forecasting method
def naive_forecast(values, horizon):

    values = np.asarray(
        values,
        dtype=float
    )

    return np.repeat(
        values[-1],
        horizon
    )


# 6. Define the seasonal naive method
def seasonal_naive_forecast(
    values,
    horizon,
    seasonal_period=12
):

    values = np.asarray(
        values,
        dtype=float
    )

    if len(values) < seasonal_period:
        raise ValueError(
            "Insufficient history for seasonal naive forecasting."
        )

    # Repeat the corresponding positions from the last seasonal cycle.
    last_cycle = values[
        -seasonal_period:
    ]

    return np.array([
        last_cycle[step % seasonal_period]
        for step in range(horizon)
    ])


# 7. Define the classic Croston method
def croston_forecast(
    values,
    horizon,
    alpha=0.1
):

    values = np.asarray(
        values,
        dtype=float
    )

    if not 0 < alpha <= 1:
        raise ValueError(
            "Croston alpha must be in the interval (0, 1]."
        )

    if (values < 0).any():
        raise ValueError(
            "Croston requires nonnegative demand."
        )

    smoothed_demand = 0.0
    smoothed_interval = 0.0
    interval = 1

    for value in values:

        if value > 0:

            if smoothed_interval == 0:

                # Initialize at the first positive observation.
                smoothed_demand = value
                smoothed_interval = float(interval)

            else:

                smoothed_demand += (
                    alpha
                    * (value - smoothed_demand)
                )

                smoothed_interval += (
                    alpha
                    * (interval - smoothed_interval)
                )

            interval = 1

        else:

            interval += 1

    if smoothed_interval > 0:

        demand_rate = (
            smoothed_demand
            / smoothed_interval
        )

    else:

        demand_rate = 0.0

    # Classic Croston produces a constant forecast over the horizon.
    return np.repeat(
        demand_rate,
        horizon
    )


# 8. Fit each baseline independently for every part
prediction_records = []
arima_diagnostics = []

for part_number, reference in enumerate(
    references,
    start=1
):

    values = (
        history[reference]
        .to_numpy(dtype=float)
    )

    forecasts = {
        "Naive": naive_forecast(
            values,
            FORECAST_HORIZON
        ),
        "SeasonalNaive": seasonal_naive_forecast(
            values,
            FORECAST_HORIZON,
            SEASONAL_PERIOD
        ),
        "Croston": croston_forecast(
            values,
            FORECAST_HORIZON,
            CROSTON_ALPHA
        )
    }

    # Fit ARIMA on the history available at the test origin.
    with warnings.catch_warnings(
        record=True
    ) as captured_warnings:

        warnings.simplefilter(
            "always"
        )

        arima_fit = ARIMA(
            values,
            order=ARIMA_ORDER
        ).fit()

        arima_prediction = np.asarray(
            arima_fit.forecast(
                steps=FORECAST_HORIZON
            ),
            dtype=float
        )

    forecasts["ARIMA_1_0_0"] = (
        arima_prediction
    )

    convergence_status = getattr(
        arima_fit,
        "mle_retvals",
        {}
    ).get(
        "converged",
        None
    )

    arima_diagnostics.append({
        "Reference": reference,
        "Converged": (
            bool(convergence_status)
            if convergence_status is not None
            else None
        ),
        "AIC": float(arima_fit.aic),
        "Warnings": " | ".join(
            str(item.message)
            for item in captured_warnings
        )
    })

    # Apply the same nonnegative forecast constraint as for the networks.
    for model_name, forecasts_array in forecasts.items():

        forecasts_array = np.maximum(
            np.asarray(
                forecasts_array,
                dtype=float
            ),
            0
        )

        if not np.isfinite(
            forecasts_array
        ).all():
            raise ValueError(
                f"Non-finite predictions for "
                f"{reference} using {model_name}."
            )

        for horizon in range(1, FORECAST_HORIZON + 1):

            target_date = (
                origin_date
                + pd.DateOffset(months=horizon)
            )

            prediction_records.append({
                "Model": model_name,
                "Reference": reference,
                "Origin_Date": origin_date.strftime(
                    "%Y-%m-%d"
                ),
                "Target_Date": target_date.strftime(
                    "%Y-%m-%d"
                ),
                "Horizon": horizon,
                "Prediction": float(
                    forecasts_array[horizon - 1]
                ),
                "Is_Synthetic": True
            })

    if (
        part_number % 12 == 0
        or part_number == len(references)
    ):

        print(
            f"Processed {part_number}/{len(references)} parts"
        )


# 9. Attach the common test targets after generating forecasts
baseline_predictions = pd.DataFrame(
    prediction_records
)

join_columns = [
    "Reference",
    "Origin_Date",
    "Target_Date",
    "Horizon"
]

baseline_predictions = baseline_predictions.merge(
    expected_targets[
        join_columns + ["Actual"]
    ],
    on=join_columns,
    how="left",
    validate="many_to_one"
)

if baseline_predictions["Actual"].isna().any():
    raise ValueError(
        "Some baseline forecasts do not match the common test targets."
    )

baseline_predictions.to_csv(
    output_folder / "baseline_test_predictions.csv",
    index=False
)

pd.DataFrame(
    arima_diagnostics
).to_csv(
    output_folder / "ARIMA_fit_diagnostics.csv",
    index=False
)


# 10. Load the neural model predictions
neural_prediction_paths = {
    "GCN": Path(
        "results/step_07/GCN_test_predictions.csv"
    ),
    "GAT": Path(
        "results/step_08/GAT_test_predictions.csv"
    ),
    "TFT": Path(
        "results/step_09/TFT_test_predictions.csv"
    )
}

prediction_tables = [
    baseline_predictions
]

for model_name, path in neural_prediction_paths.items():

    model_predictions = pd.read_csv(
        path
    )

    if not model_predictions["Model"].eq(
        model_name
    ).all():
        raise ValueError(
            f"Unexpected model labels in {path}"
        )

    prediction_tables.append(
        model_predictions
    )

# Include the score ablation if Code 12 has been executed.
ablation_path = Path(
    "results/step_12/"
    "GCN_without_score_test_predictions.csv"
)

if ablation_path.exists():

    prediction_tables.append(
        pd.read_csv(ablation_path)
    )

all_predictions = pd.concat(
    prediction_tables,
    ignore_index=True
)


# 11. Verify identical test identities for every model
identity_columns = (
    join_columns + ["Actual"]
)

expected_identities = (
    expected_targets[identity_columns]
    .sort_values(["Reference", "Horizon"])
    .reset_index(drop=True)
)

for model_name, group in all_predictions.groupby("Model"):

    if group.duplicated(
        join_columns
    ).any():
        raise ValueError(
            f"Duplicate test forecasts for {model_name}."
        )

    actual_identities = (
        group[identity_columns]
        .sort_values(["Reference", "Horizon"])
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        actual_identities,
        expected_identities,
        check_dtype=False,
        check_exact=False,
        rtol=1e-6,
        atol=1e-6
    )


# 12. Calculate the extended comparison metrics
comparison_records = []

for model_name, group in all_predictions.groupby("Model"):

    comparison_records.append({
        "Model": model_name,
        **calculate_metrics(
            group["Actual"],
            group["Prediction"]
        )
    })

comparison_df = (
    pd.DataFrame(comparison_records)
    .sort_values("MAE")
    .reset_index(drop=True)
)

comparison_df.to_csv(
    output_folder / "extended_model_comparison.csv",
    index=False
)

all_predictions.to_csv(
    output_folder / "all_test_predictions.csv",
    index=False
)

print("\nExtended model comparison, sorted by MAE:")
print(
    comparison_df.round(4).to_string(index=False)
)


# 13. Calculate metrics separately for each horizon
horizon_records = []

for (model_name, horizon), group in (
    all_predictions.groupby(
        ["Model", "Horizon"]
    )
):

    horizon_records.append({
        "Model": model_name,
        "Horizon": int(horizon),
        **calculate_metrics(
            group["Actual"],
            group["Prediction"]
        )
    })

pd.DataFrame(
    horizon_records
).to_csv(
    output_folder / "extended_comparison_by_horizon.csv",
    index=False
)


# 14. Plot the extended comparison
plot_df = comparison_df.sort_values(
    "MAE",
    ascending=False
)

neural_names = {
    "GCN",
    "GAT",
    "TFT",
    "GCN_without_score"
}

bar_colors = [
    "#4472C4" if name in neural_names else "#ED7D31"
    for name in plot_df["Model"]
]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5)
)

for ax, metric in zip(
    axes,
    ["MAE", "WAPE_pct"]
):

    ax.barh(
        plot_df["Model"],
        plot_df[metric],
        color=bar_colors
    )

    ax.set_xlabel(
        "MAE" if metric == "MAE" else "WAPE (%)"
    )

    ax.set_title(
        "Lower values indicate lower error"
    )

    ax.grid(
        axis="x",
        alpha=0.25
    )

fig.suptitle(
    "Neural models and statistical baselines"
)

fig.tight_layout()

fig.savefig(
    output_folder / "extended_model_comparison.png",
    dpi=160
)

plt.show()
plt.close(fig)


# 15. Save the baseline evaluation protocol
protocol = {
    "forecast_origin": origin_date.strftime("%Y-%m-%d"),
    "forecast_horizon": FORECAST_HORIZON,
    "seasonal_period": SEASONAL_PERIOD,
    "croston_method": "Classic Croston",
    "croston_alpha": CROSTON_ALPHA,
    "arima_order": list(ARIMA_ORDER),
    "baseline_fitting": (
        "Each baseline uses the history available "
        "up to and including the test origin."
    ),
    "neural_model_fitting": (
        "Neural weights were trained on the training period "
        "and selected using validation data. Their historical "
        "test inputs extend to the same test origin."
    ),
    "arima_diagnostics_file": "ARIMA_fit_diagnostics.csv",
    "test_targets_identical": True,
    "nonnegative_forecasts": True,
    "models_evaluated": comparison_df["Model"].tolist(),
    "interpretation": (
        "These are synthetic benchmark results. "
        "No model superiority or significance is assumed."
    )
}

(
    output_folder / "baseline_protocol.json"
).write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8"
)

print(f"\nResults saved in: {output_folder}")